## 1) 실행 환경 및 라이브러리 준비
이 단계는 분석 재현성을 위해 파이썬 환경과 시각화 설정을 먼저 고정합니다.


In [ ]:
# 현재 노트북이 어떤 파이썬 인터프리터에서 실행되는지 확인합니다.
# (환경이 다르면 패키지/결과가 달라질 수 있어 재현성 점검용)
import sys
print(sys.executable)

In [ ]:
import re
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

## 2) 데이터베이스 연결 및 원천 데이터 로드
분석에 필요한 소음/민원/진동/인구 데이터를 DB에서 불러옵니다.


In [ ]:
# DB 접속 정보를 설정해 분석용 원천 테이블에 접근합니다.
# 연결 성공 여부를 먼저 확인해 이후 셀 에러를 예방합니다.
from sqlalchemy import create_engine
import pandas as pd

user = "team8_admin"
password = "admin1!"
host = "34.50.4.76"
port = 3306
database = "team8_11"

engine = create_engine(
    f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"
)

conn = engine.connect()
print("연결 성공!")

In [ ]:
# 분석에 필요한 테이블을 한 번에 로드합니다.
# 인구는 2022~2024 컬럼만 선택해 후속 계산 컬럼을 단순화합니다.
noise = pd.read_sql("SELECT * FROM noise_integrated", engine)
noise_com = pd.read_sql("SELECT * FROM noise_complaint", engine)
vibration = pd.read_sql("SELECT * FROM vibration_integrated", engine)

crime = pd.read_sql("SELECT * FROM 5_crimes", engine)
dispatch = pd.read_sql("SELECT * FROM dispatch_integrated", engine)

population = pd.read_sql(
    """
    SELECT 자치구,
           `2022_행정구역(A)_인구 (명)`,
           `2023_행정구역(A)_인구 (명)`,
           `2024_행정구역(A)_인구 (명)`
    FROM garbage_collection_status
    """,
    engine
)


In [ ]:
noise.columns, noise_com.columns, vibration.columns, population.columns

In [ ]:
noise_com.head()

In [ ]:
noise_com.isna().sum()

In [ ]:
noise.isna().sum()

In [ ]:
vibration.isna().sum()

In [ ]:
population.head()

## 민원점수

- 소음진동 민원 데이터와 인구 데이터를 자치구 기준으로 결합
- 단순 민원 건수는 인구가 많은 자치구에 불리하므로, 인구 1만 명당 민원 건수로 보정
- 이후 연도별 1~8점 범위의 민원 스트레스 점수를 만듬

## 3) 민원 데이터 전처리 및 인구 보정 지표 생성
자치구 단위 비교가 가능하도록 컬럼을 정리하고, 인구 1만 명당 민원으로 보정합니다.


In [ ]:
# 컬럼명을 분석 친화적으로 변경합니다.
# 연도별 의미가 드러나도록 이름을 명시해 가독성과 유지보수성을 높입니다.
# 컬럼 이름 변경
noise_com = noise_com.rename(columns={
    "자치구별(2)": "자치구",
    "2022": "환경관련민원_2022",
    "2022_1": "소음진동민원_2022",
    "2023": "환경관련민원_2023",
    "2023_1": "소음진동민원_2023",
    "2024": "환경관련민원_2024",
    "2024_1": "소음진동민원_2024",
})

population = population.rename(columns={
    "2022_행정구역(A)_인구 (명)": "인구_2022",
    "2023_행정구역(A)_인구 (명)": "인구_2023",
    "2024_행정구역(A)_인구 (명)": "인구_2024",
})


In [ ]:
# '소계'는 자치구 비교를 왜곡하므로 제외합니다.
# 구 단위 상대 비교가 목적이기 때문에 합계 행은 제거합니다.
# 구별끼리의 비교라 소계는 제외
noise_com = noise_com[noise_com["자치구"] != "소계"].copy()
population = population[population["자치구"] != "소계"].copy()

In [ ]:
noise_com["자치구"].nunique()

In [ ]:
population["자치구"].nunique()

In [ ]:
# 자치구 기준으로 민원 + 인구를 결합합니다.
# 이후 '인구 보정 민원' 계산의 기준 테이블입니다.
# 자치구별 민원 데이터와 인구 데이터를 합침
complaint_pop = noise_com[
    ["자치구", "소음진동민원_2022", "소음진동민원_2023", "소음진동민원_2024"]
].merge(
    population[["자치구", "인구_2022", "인구_2023", "인구_2024"]],
    on="자치구",
    how="left"
)

complaint_pop.head()

In [ ]:
complaint_pop.isnull().sum()


In [ ]:
# 단순 민원 건수 대신 인구 1만 명당 민원을 계산합니다.
# 인구 규모가 다른 자치구 간 비교를 공정하게 맞추기 위한 보정입니다.
# 인구만명당 소음민원
complaint_pop["만명당소음민원_2022"] = complaint_pop["소음진동민원_2022"] / complaint_pop["인구_2022"] * 10000
complaint_pop["만명당소음민원_2023"] = complaint_pop["소음진동민원_2023"] / complaint_pop["인구_2023"] * 10000
complaint_pop["만명당소음민원_2024"] = complaint_pop["소음진동민원_2024"] / complaint_pop["인구_2024"] * 10000

complaint_pop

In [ ]:
# 3개년 평균으로 단년도 변동 노이즈를 줄이고 구조적 수준을 봅니다.
complaint_pop["만명당소음민원_평균"] = complaint_pop[
    ["만명당소음민원_2022", "만명당소음민원_2023", "만명당소음민원_2024"]
].mean(axis=1)

complaint_pop.sort_values('만명당소음민원_평균', ascending=False)

In [ ]:
# 전년 대비 절대 증감량을 계산해 변화 방향을 확인합니다.
complaint_pop["증감_22_23"] = complaint_pop["만명당소음민원_2023"] - complaint_pop["만명당소음민원_2022"]
complaint_pop["증감_23_24"] = complaint_pop["만명당소음민원_2024"] - complaint_pop["만명당소음민원_2023"]

complaint_pop[[
    "자치구",
    "만명당소음민원_2022",
    "만명당소음민원_2023",
    "만명당소음민원_2024",
    "만명당소음민원_평균",
    "증감_22_23",
    "증감_23_24"
]]


In [ ]:
# 절대 증감량만으로는 해석이 어려워 증감률(%)도 함께 계산합니다.
complaint_pop["증감률_22_23"] = (
    complaint_pop["증감_22_23"] / complaint_pop["만명당소음민원_2022"]
) * 100

complaint_pop["증감률_23_24"] = (
    complaint_pop["증감_23_24"] / complaint_pop["만명당소음민원_2023"]
) * 100

complaint_pop

## 4) 민원 지표 시각화 및 스트레스 점수화
연도별 패턴을 시각화한 뒤, 1~8점 스케일로 변환해 해석 가능성을 높입니다.


In [ ]:
# 자치구 x 연도 히트맵으로 고/저 민원 구역을 한눈에 확인합니다.
heatmap_df = complaint_pop.set_index("자치구")[
    ["만명당소음민원_2022", "만명당소음민원_2023", "만명당소음민원_2024"]
]
heatmap_df.columns = ["2022", "2023", "2024"]


plt.figure(figsize=(8, 10))
sns.heatmap(
    heatmap_df,
    annot=True,
    fmt=".1f",
    cmap="Reds"
)
plt.title("자치구별 연도별 인구 1만 명당 소음진동 민원")
plt.xlabel("연도")
plt.ylabel("자치구")
plt.show()

In [ ]:
years = [2022, 2023, 2024]

for year in years:
    col = f"만명당소음민원_{year}"
    plot_df = complaint_pop.sort_values(col, ascending=False)

    plt.figure(figsize=(10, 8))
    ax = sns.barplot(
        data=plot_df,
        x=col,
        y="자치구",
        palette="Reds_r"
    )
    ax.bar_label(ax.containers[0], fmt="%.1f")
    plt.title(f"{year}년 자치구별 인구 1만 명당 소음진동 민원")
    plt.xlabel("1만 명당 소음진동 민원")
    plt.ylabel("자치구")
    plt.tight_layout()
    plt.show()

In [ ]:
# 0점 해석 문제를 피하기 위해 1~8점으로 선형 환산합니다.
# 동일값만 존재하면 중간점수(4.5)로 처리해 분모 0 문제를 방지합니다.
# 스트레스가 전혀 없는 것처럼 보이는 0점을 피하기 위해 1~8점 범위로 환산
# 값이 클수록 스트레스가 높음
def to_1_8_stress_score(s):
    s = pd.to_numeric(s, errors="coerce")
    mn = s.min()
    mx = s.max()
    if mx == mn:
        return pd.Series(4.5, index=s.index)
    return 1 + (s - mn) / (mx - mn) * 7

In [ ]:
# 연도별 민원지표를 동일 스케일(1~8)로 변환해 비교 가능하게 만듭니다.
complaint_pop["민원스트레스점수_2022"] = to_1_8_stress_score(complaint_pop["만명당소음민원_2022"]).round(2)
complaint_pop["민원스트레스점수_2023"] = to_1_8_stress_score(complaint_pop["만명당소음민원_2023"]).round(2)
complaint_pop["민원스트레스점수_2024"] = to_1_8_stress_score(complaint_pop["만명당소음민원_2024"]).round(2)


In [ ]:
complaint_pop[[
    "자치구",
    "만명당소음민원_2022", "민원스트레스점수_2022",
    "만명당소음민원_2023", "민원스트레스점수_2023",
    "만명당소음민원_2024", "민원스트레스점수_2024"
]].head()


In [ ]:
import pandas as pd
import plotly.express as px

seoul_coords = {
    "종로구": (37.573520, 126.978834),
    "중구": (37.563843, 126.997602),
    "용산구": (37.532527, 126.990490),
    "성동구": (37.563341, 127.036289),
    "광진구": (37.538617, 127.082375),
    "동대문구": (37.574368, 127.039593),
    "중랑구": (37.606324, 127.092584),
    "성북구": (37.589400, 127.016749),
    "강북구": (37.639610, 127.025657),
    "도봉구": (37.668768, 127.047163),
    "노원구": (37.654191, 127.056793),
    "은평구": (37.617612, 126.922700),
    "서대문구": (37.579115, 126.936778),
    "마포구": (37.566324, 126.901491),
    "양천구": (37.516873, 126.866399),
    "강서구": (37.550979, 126.849538),
    "구로구": (37.495485, 126.887445),
    "금천구": (37.456864, 126.895510),
    "영등포구": (37.526371, 126.896229),
    "동작구": (37.512402, 126.939252),
    "관악구": (37.478406, 126.951613),
    "서초구": (37.483712, 127.032411),
    "강남구": (37.517305, 127.047502),
    "송파구": (37.514575, 127.105399),
    "강동구": (37.530126, 127.123770),
}


In [ ]:
# 지도 시각화를 위해 자치구별 위/경도를 결합합니다.
coord_df = pd.DataFrame([
    {"자치구": gu, "lat": lat, "lon": lon}
    for gu, (lat, lon) in seoul_coords.items()
])

map_df = complaint_pop.merge(coord_df, on="자치구", how="left")
map_df


In [ ]:
len(map_df)

In [ ]:
# 지도에서 점수의 크기/색으로 공간 분포를 직관적으로 비교합니다.
for year in [2022, 2023, 2024]:
    fig = px.scatter_mapbox(
        map_df,
        lat="lat",
        lon="lon",
        size=f"민원스트레스점수_{year}",
        color=f"민원스트레스점수_{year}",
        hover_name="자치구",
        hover_data={
            "lat": False,
            "lon": False,
            f"만명당소음민원_{year}": ":.2f",
            f"민원스트레스점수_{year}": ":.2f",
        },
        color_continuous_scale="Reds",
        range_color=(0, 8),
        size_max=30,
        zoom=10,
        center={"lat": 37.55, "lon": 126.98},
        title=f"{year} 자치구별 민원 스트레스 점수"
    )
    fig.update_layout(mapbox_style="carto-positron")
    fig.show()



## 소음점수


## 5) 소음 계측 데이터 가공 및 대표지표 산출
월별 측정값을 자치구/연도 단위로 집계해 소음 강도를 종합 지표로 만듭니다.


In [ ]:
noise.head()

In [ ]:
# wide 형식 월별 컬럼을 long 형식으로 변환해 집계/분석을 쉽게 만듭니다.
# 연도/월 파생 컬럼을 만들어 시계열 집계를 준비합니다.

noise_long = noise.melt(
    id_vars=["gu_code", "자치구", "지역"],
    var_name="연월",
    value_name="소음값"
)

noise_long["소음값"] = pd.to_numeric(noise_long["소음값"], errors="coerce")
noise_long["연도"] = "20" + noise_long["연월"].str[:2]
noise_long["월"] = noise_long["연월"].str[2:4].astype(int)

noise_long.head()

In [ ]:
noise_long.groupby("자치구")["지역"].nunique().sort_values(ascending=False)

In [ ]:
noise_long["연도"].value_counts()

In [ ]:
# 자치구·연도 평균소음: 전반적 노출 수준을 대표합니다.

gu_year_noise_mean = noise_long.groupby(["자치구", "연도"], as_index=False)["소음값"].mean()
gu_year_noise_mean = gu_year_noise_mean.rename(columns={"소음값": "평균소음"})
gu_year_noise_mean

In [ ]:
# 자치구·연도 최고소음: 피크(극단값) 위험을 반영합니다.
gu_year_noise_max = noise_long.groupby(["자치구", "연도"], as_index=False)["소음값"].max()
gu_year_noise_max = gu_year_noise_max.rename(columns={"소음값": "최고소음"})
gu_year_noise_max

In [ ]:
# 자치구·연도 소음편차: 변동성(불안정성) 정보를 반영합니다.
gu_year_noise_std = noise_long.groupby(["자치구", "연도"], as_index=False)["소음값"].std()
gu_year_noise_std = gu_year_noise_std.rename(columns={"소음값": "소음편차"})
gu_year_noise_std

In [ ]:
noise_summary = gu_year_noise_mean.merge(
    gu_year_noise_max,
    on=["자치구", "연도"],
    how="left"
).merge(
    gu_year_noise_std,
    on=["자치구", "연도"],
    how="left"
)

noise_summary.head()

In [ ]:
# 지표 단위가 달라 직접합이 어렵기 때문에 z-score 표준화를 적용합니다.
def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    std = s.std()
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=s.index)
    return (s - s.mean()) / std

In [ ]:
# 평균(0.6), 최고(0.3), 편차(0.1) 가중합으로 소음대표지표를 구성합니다.
# 평균에 높은 가중치를 둬 '지속적 소음 노출' 영향을 더 크게 반영합니다.

noise_summary["소음대표지표"] = (
    0.6 * zscore(noise_summary["평균소음"]) +
    0.3 * zscore(noise_summary["최고소음"]) +
    0.1 * zscore(noise_summary["소음편차"].fillna(0))
)

noise_summary.head()


In [ ]:
def to_1_8_stress_score(s):
    s = pd.to_numeric(s, errors="coerce")
    mn = s.min()
    mx = s.max()
    if mx == mn:
        return pd.Series(4.5, index=s.index)
    return 1 + (s - mn) / (mx - mn) * 7

In [ ]:
# 연도별 분포 기준으로 1~8점 환산해 연도 간 상대 비교를 쉽게 만듭니다.
noise_summary["소음스트레스점수"] = noise_summary.groupby("연도")["소음대표지표"].transform(
    lambda x: to_1_8_stress_score(x)
).round(2)

noise_summary.head()

In [ ]:
noise_summary["소음스트레스점수"] = noise_summary.groupby("연도")["소음대표지표"].transform(
    lambda x: to_1_8_stress_score(x)
).round(2)

noise_summary

In [ ]:
noise_score_wide = noise_summary.pivot(
    index="자치구",
    columns="연도",
    values="소음스트레스점수"
).reset_index()

noise_score_wide.columns = ["자치구", "소음스트레스점수_2022", "소음스트레스점수_2023", "소음스트레스점수_2024"]

noise_score_wide.head()

In [ ]:
years = [2022, 2023, 2024]

for year in years:
    col = f"소음스트레스점수_{year}"
    plot_df = noise_score_wide.sort_values(col, ascending=False)

    plt.figure(figsize=(10, 8))
    ax = sns.barplot(
        data=plot_df,
        x=col,
        y="자치구",
        palette="Purples_r"
    )
    ax.bar_label(ax.containers[0], fmt="%.1f")
    plt.title(f"{year}년 자치구별 소음 스트레스 점수")
    plt.xlabel("소음 스트레스 점수")
    plt.ylabel("자치구")
    plt.xlim(0, 8.5)
    plt.tight_layout()
    plt.show()


In [ ]:
years = ["2022", "2023", "2024"]

for year in years:
    plot_df = noise_summary[noise_summary["연도"] == year].sort_values("소음대표지표", ascending=False)

    plt.figure(figsize=(10, 8))
    ax = sns.barplot(
        data=plot_df,
        x="소음대표지표",
        y="자치구",
        palette="Purples_r"
    )
    ax.bar_label(ax.containers[0], fmt="%.2f")
    plt.title(f"{year}년 자치구별 소음 대표지표")
    plt.xlabel("소음 대표지표")
    plt.ylabel("자치구")
    plt.tight_layout()
    plt.show()


## 진동점수

## 6) 진동 데이터 가공 및 축별 점수 통합
X/Y/Z 축별 진동을 점수화하고 평균해 진동 스트레스 점수를 만듭니다.


In [ ]:
# 진동 데이터도 long 형식으로 변환하고 연도/월/축 정보를 분리합니다.

vibration_long = vibration.melt(
    id_vars=["gu_code", "자치구"],
    var_name="연월축",
    value_name="진동값"
)

vibration_long["진동값"] = pd.to_numeric(vibration_long["진동값"], errors="coerce")
vibration_long["연도"] = "20" + vibration_long["연월축"].str[:2]
vibration_long["월"] = vibration_long["연월축"].str[2:4].astype(int)
vibration_long["축"] = vibration_long["연월축"].str[-1]

vibration_long.head()

In [ ]:

vibration_axis_summary = vibration_long.groupby(["자치구", "연도", "축"], as_index=False)["진동값"].mean()
vibration_axis_summary = vibration_axis_summary.rename(columns={"진동값": "평균진동"})
vibration_axis_summary

In [ ]:
# 축별(X/Y/Z)로 분포가 다를 수 있어 연도·축 단위로 별도 점수화합니다.
vibration_axis_summary["축별진동스트레스점수"] = vibration_axis_summary.groupby(["연도", "축"])["평균진동"].transform(
    lambda x: to_1_8_stress_score(x)
).round(2)
vibration_axis_summary

In [ ]:
vibration_axis_wide = vibration_axis_summary.pivot(
    index=["자치구", "연도"],
    columns="축",
    values="축별진동스트레스점수"
).reset_index()

vibration_axis_wide.columns = ["자치구", "연도", "진동X점수", "진동Y점수", "진동Z점수"]
vibration_axis_wide

In [ ]:
# 축별 점수 평균으로 자치구의 진동 종합 스트레스 점수를 만듭니다.

vibration_axis_wide["진동스트레스점수"] = vibration_axis_wide[
    ["진동X점수", "진동Y점수", "진동Z점수"]
].mean(axis=1).round(2)

vibration_axis_wide.head()

In [ ]:
vibration_score_wide = vibration_axis_wide.pivot(
    index="자치구",
    columns="연도",
    values="진동스트레스점수"
).reset_index()

vibration_score_wide.columns = [
    "자치구",
    "진동스트레스점수_2022",
    "진동스트레스점수_2023",
    "진동스트레스점수_2024"
]

vibration_score_wide.head()


## 7) 최종 소음 총점 계산
민원·소음·진동 점수를 합쳐 연도별 최종 점수를 산출합니다.


In [ ]:
# 민원/소음/진동 점수를 자치구 기준으로 결합해 최종 점수 계산 기반을 만듭니다.

final_noise = complaint_pop[[
    "자치구",
    "민원스트레스점수_2022",
    "민원스트레스점수_2023",
    "민원스트레스점수_2024"
]].merge(
    noise_score_wide,
    on="자치구",
    how="left"
).merge(
    vibration_score_wide,
    on="자치구",
    how="left"
)

final_noise.head()

In [ ]:
final_noise.isnull().sum()

In [ ]:
# 3개 축(민원·소음·진동)을 합산해 소음원점수(최대 24점)를 계산합니다.
final_noise["소음원점수_2022"] = (
    final_noise["민원스트레스점수_2022"] +
    final_noise["소음스트레스점수_2022"] +
    final_noise["진동스트레스점수_2022"]
).round(2)

final_noise["소음원점수_2023"] = (
    final_noise["민원스트레스점수_2023"] +
    final_noise["소음스트레스점수_2023"] +
    final_noise["진동스트레스점수_2023"]
).round(2)

final_noise["소음원점수_2024"] = (
    final_noise["민원스트레스점수_2024"] +
    final_noise["소음스트레스점수_2024"] +
    final_noise["진동스트레스점수_2024"]
).round(2)

In [ ]:
# 24점 체계를 25점 만점으로 선형 환산해 최종 지표 스케일을 통일합니다.
final_noise["소음총점수_2022"] = (final_noise["소음원점수_2022"] * (25 / 24)).round(2)
final_noise["소음총점수_2023"] = (final_noise["소음원점수_2023"] * (25 / 24)).round(2)
final_noise["소음총점수_2024"] = (final_noise["소음원점수_2024"] * (25 / 24)).round(2)

In [ ]:
final_noise[[
    "자치구",
    "민원스트레스점수_2022", "소음스트레스점수_2022", "진동스트레스점수_2022", "소음총점수_2022",
    "민원스트레스점수_2023", "소음스트레스점수_2023", "진동스트레스점수_2023", "소음총점수_2023",
    "민원스트레스점수_2024", "소음스트레스점수_2024", "진동스트레스점수_2024", "소음총점수_2024"
]].head()

In [ ]:
final_noise.sort_values("소음총점수_2022", ascending=False)[
    ["자치구", "소음총점수_2022"]
]

In [ ]:
# 연도별 최종 점수 순위를 시각화해 정책 우선순위 후보를 확인합니다.
years = [2022, 2023, 2024]

for year in years:
    col = f"소음총점수_{year}"
    plot_df = final_noise.sort_values(col, ascending=False)

    plt.figure(figsize=(10, 8))
    ax = sns.barplot(
        data=plot_df,
        x=col,
        y="자치구",
        palette="Purples_r"
    )
    ax.bar_label(ax.containers[0], fmt="%.1f")
    plt.title(f"{year}년 자치구별 소음총점수")
    plt.xlabel("소음총점수 (25점 만점)")
    plt.ylabel("자치구")
    plt.xlim(0, 26)
    plt.tight_layout()
    plt.show()


## 8) 지도 시각화
최종 점수를 지도 위에서 직관적으로 비교할 수 있도록 시각화합니다.


In [ ]:
image_path = "SUSI.png"

In [ ]:
import pandas as pd

seoul_coords = {
    "종로구": (37.573520, 126.978834),
    "중구": (37.563843, 126.997602),
    "용산구": (37.532527, 126.990490),
    "성동구": (37.563341, 127.036289),
    "광진구": (37.538617, 127.082375),
    "동대문구": (37.574368, 127.039593),
    "중랑구": (37.606324, 127.092584),
    "성북구": (37.589400, 127.016749),
    "강북구": (37.639610, 127.025657),
    "도봉구": (37.668768, 127.047163),
    "노원구": (37.654191, 127.056793),
    "은평구": (37.617612, 126.922700),
    "서대문구": (37.579115, 126.936778),
    "마포구": (37.566324, 126.901491),
    "양천구": (37.516873, 126.866399),
    "강서구": (37.550979, 126.849538),
    "구로구": (37.495485, 126.887445),
    "금천구": (37.456864, 126.895510),
    "영등포구": (37.526371, 126.896229),
    "동작구": (37.512402, 126.939252),
    "관악구": (37.478406, 126.951613),
    "서초구": (37.483712, 127.032411),
    "강남구": (37.517305, 127.047502),
    "송파구": (37.514575, 127.105399),
    "강동구": (37.530126, 127.123770),
}

coord_df = pd.DataFrame([
    {"자치구": gu, "lat": lat, "lon": lon}
    for gu, (lat, lon) in seoul_coords.items()
])

total_map_df = final_noise.merge(coord_df, on="자치구", how="left")
total_map_df.head()


In [ ]:
# 커스텀 아이콘 크기를 점수와 연동해 지도에서 체감 차이를 강조합니다.
import folium
from folium.features import CustomIcon

for year in [2022, 2023, 2024]:
    score_col = f"소음총점수_{year}"
    
    m = folium.Map(location=[37.55, 126.98], zoom_start=11, tiles="CartoDB positron")
    
    for _, row in total_map_df.iterrows():
        score = row[score_col]
        icon_size = int(25 + score * 2.2)

        icon = CustomIcon(
            icon_image=image_path,
            icon_size=(icon_size, icon_size),
            icon_anchor=(icon_size // 2, icon_size // 2)
        )

        popup_text = f"""
        <b>{row['자치구']}</b><br>
        {year} 소음총점수: {score:.2f}
        """

        folium.Marker(
            location=[row["lat"], row["lon"]],
            icon=icon,
            popup=folium.Popup(popup_text, max_width=250),
            tooltip=f"{row['자치구']} / {score:.2f}"
        ).add_to(m)
    
    display(m)

In [ ]:
# 최소/최대 점수 기반 정규화 크기 스케일로 연도별 시각적 왜곡을 줄입니다.
import folium
from folium.features import CustomIcon

for year in [2022, 2023, 2024]:
    score_col = f"소음총점수_{year}"
    
    m = folium.Map(location=[37.55, 126.98], zoom_start=11, tiles="CartoDB positron")
    
    min_size = 10
    max_size = 100

    min_score = total_map_df[score_col].min()
    max_score = total_map_df[score_col].max()
    
    for _, row in total_map_df.iterrows():
        score = row[score_col]

        icon_size = int(
            min_size + (score - min_score) / (max_score - min_score) * (max_size - min_size)
        )

        icon = CustomIcon(
            icon_image=image_path,
            icon_size=(icon_size, icon_size),
            icon_anchor=(icon_size // 2, icon_size // 2)
        )

        popup_text = f"""
        <b>{row['자치구']}</b><br>
        {year} 소음총점수: {score:.2f}
        """

        folium.Marker(
            location=[row["lat"], row["lon"]],
            icon=icon,
            popup=folium.Popup(popup_text, max_width=250),
            tooltip=f"{row['자치구']} / {score:.2f}"
        ).add_to(m)
    
    display(m)


In [ ]:
final_noise.to_csv("소음최종점수_2022_2024.csv", index=False, encoding="utf-8-sig")